# Аугментация слабых классов (ЖКТ + КОЖА): baseline vs трансформер

Перефразируем жалобы двух слабых классов через Mistral, дообучаем на этом
baseline и RuBioRoBERTa, сравниваем на dev с результатами без аугментации
(baseline 0.841, трансформер 0.854).

**Перед запуском:** Runtime → Change runtime type → GPU.

In [ ]:
%cd /content
!rm -rf anamnesis && git clone -q https://github.com/vinograd131/anamnesis.git
%cd anamnesis
!pip install -q mistralai transformers accelerate peft
!pip uninstall -q -y torchao

In [ ]:
import getpass, os
os.environ["MISTRAL_API_KEY"] = getpass.getpass("MISTRAL_API_KEY: ")

## 1. Аугментация слабых классов

Добиваем ЖКТ и КОЖА до 500 примеров перефразировками (создаёт `data/train_aug_v1.jsonl`).

In [ ]:
from src.augment import main as augment
augment(target_per_class=500, groups=["ЖКТ/БРЮШНЫЕ", "КОЖА"])

## 2. Baseline (tf-idf + LogReg) на аугментированном train

Сравнение с baseline без аугментации: dev macro-F1 0.841.

In [ ]:
from src.baseline import build_model
from src.data import load_xy
from src.evaluate import scores, print_report
from src.mapping import GROUPS

x_tr, y_tr = load_xy("train_aug")
x_dv, y_dv = load_xy("dev")
clf = build_model().fit(x_tr, y_tr)
pred = clf.predict(x_dv)
print("baseline+aug on dev:", scores(y_dv, pred))
print_report(y_dv, pred, labels=list(GROUPS))

## 3. Трансформер (RuBioRoBERTa) на аугментированном train

Дообучение с лучшими параметрами Optuna. Сравнение с трансформером без аугментации: dev macro-F1 0.854.

In [ ]:
from src.transformer_ft_aug import train
train(eval_split="dev")

## Метрики

Файлы в Colab временные — скачай результаты или запушь в git.

In [ ]:
import json
print(json.dumps(json.load(open("reports/metrics.json")), ensure_ascii=False, indent=2))

## Сохранить аугментацию и результаты в git (опционально)

Чтобы `train_aug` и метрики не потерялись. Нужен GitHub-токен (вводится скрыто).

In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
!git config user.email "karinkakarik@mail.ru"
!git config user.name "Git_Karina"
!git add data/train_aug_v1.jsonl reports/
!git commit -q -m "augmentation results"
!git pull -q --rebase https://github.com/vinograd131/anamnesis.git main
!git push -q https://{token}@github.com/vinograd131/anamnesis.git HEAD:main